In [22]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

In [23]:
test_data = pd.read_csv("../output/test_predictions.csv")
og_test_data = pd.read_csv("../data/split/test.csv", index_col=0)

In [24]:
val_data = pd.read_csv("../output/val_predictions.csv")
og_val_data = pd.read_csv("../data/split/val.csv", index_col=0)

In [25]:
inf_data = pd.read_csv("../output/predictions.csv")
og_inf_data = pd.read_csv("../data/test.csv", index_col=0)

In [26]:
new_data = pd.merge(val_data, og_val_data, on="row_id")
new_test_data = pd.merge(test_data, og_test_data, on="row_id")
new_inf_data = pd.merge(inf_data, og_inf_data, on="row_id")

In [27]:
def parse_bbox(x):
    x1, y1, w, h = x[1:-1].split(",")
    return float(x1), float(y1), float(w), float(h)

In [28]:
def extract_coordinates(new_data):
    x1s = []
    y1s = []
    ws = []
    hs = []
    for row in new_data["bbox"]:
        x1, y1, w, h = parse_bbox(row)
        x1s.append(x1)
        y1s.append(y1)
        ws.append(w)
        hs.append(h)
    return x1s, y1s, ws, hs

In [29]:
#Val
x1s, y1s, ws, hs = extract_coordinates(new_data)
new_data["y"] = y1s
new_data["w"] = ws
new_data["x"] = x1s
new_data["h"] = hs
new_data["w"] = new_data["w"] / new_data["width"]
new_data["x"] = new_data["x"] / new_data["width"]
new_data["h"] = new_data["h"] / new_data["height"]
new_data["y"] = new_data["y"] / new_data["height"]
new_data["wh"] = new_data["w"] / new_data["h"]

#Test
x1s, y1s, ws, hs = extract_coordinates(new_test_data)
new_test_data["y"] = y1s
new_test_data["w"] = ws
new_test_data["x"] = x1s
new_test_data["h"] = hs
new_test_data["w"] = new_test_data["w"] / new_test_data["width"]
new_test_data["x"] = new_test_data["x"] / new_test_data["width"]
new_test_data["h"] = new_test_data["h"] / new_test_data["height"]
new_test_data["y"] = new_test_data["y"] / new_test_data["height"]
new_test_data["wh"] = new_test_data["w"] / new_test_data["h"]

# Inf

x1s, y1s, ws, hs = extract_coordinates(new_inf_data)
new_inf_data["y"] = y1s
new_inf_data["w"] = ws
new_inf_data["x"] = x1s
new_inf_data["h"] = hs
new_inf_data["w"] = new_inf_data["w"] / new_inf_data["width"]
new_inf_data["x"] = new_inf_data["x"] / new_inf_data["width"]
new_inf_data["h"] = new_inf_data["h"] / new_inf_data["height"]
new_inf_data["y"] = new_inf_data["y"] / new_inf_data["height"]
new_inf_data["wh"] = new_inf_data["w"] / new_inf_data["h"]

In [8]:
x_train = new_data.iloc[:, [1, 2, 3, 4, 5, 11, 12, 13, 14, 15]]
y_train = new_data["class_id"]

In [9]:
x_test = new_test_data.iloc[:, [1, 2, 3, 4, 5, 11, 12, 13, 14, 15]]
y_test = new_test_data["class_id"]

In [32]:
x_inf = new_inf_data.iloc[:, [1, 2, 3, 4, 5, 10, 11, 12, 13, 14]]

In [34]:
model = RandomForestClassifier(random_state=42)

In [35]:
model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [36]:
predictions = model.predict(x_test)

In [37]:
accuracy_score(y_test, predictions)

0.946058091286307

In [38]:
f1_score(y_test, predictions, average="macro")

0.9106322410188519

In [39]:
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

[[303   0   0   0  14]
 [  2 331   0   1  20]
 [  3   1  47  11   8]
 [  0   2   2 991  15]
 [ 16  21   2  12 608]]
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       317
           1       0.93      0.94      0.93       354
           2       0.92      0.67      0.78        70
           3       0.98      0.98      0.98      1010
           4       0.91      0.92      0.92       659

    accuracy                           0.95      2410
   macro avg       0.94      0.89      0.91      2410
weighted avg       0.95      0.95      0.95      2410



In [40]:
predictions = model.predict(x_inf)

In [44]:
inf_data["class_id"] = predictions

In [48]:
inf_data[["row_id", "class_id"]].to_csv("predictions_rf.csv", index=False)